# 05주차 2번 과제 — VGG16 전이 학습

CIFAR-10 (cat/dog)에서 VGG16 **랜덤 초기화(Scratch)** vs **전이 학습(Transfer)** 성능 비교

In [ ]:
%matplotlib inline
import time
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.models import VGG16_Weights

print(f"torch={torch.__version__}  device={'cuda' if torch.cuda.is_available() else 'cpu'}")

## Config

In [ ]:
DATA_DIR   = "/content/data"
EPOCHS     = 5
BATCH_SIZE = 32
LR         = 1e-3
TRAIN_N    = 200
VAL_N      = 50
TEST_N     = 50
SEED       = 42
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_CAT, _DOG = 3, 5

torch.manual_seed(SEED)

## Dataset

In [ ]:
def get_transform(train):
    aug = [transforms.RandomHorizontalFlip()] if train else []
    return transforms.Compose([
        transforms.Resize(224), *aug,
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

class CatDogCIFAR(Dataset):
    def __init__(self, train, transform):
        base = datasets.CIFAR10(DATA_DIR, train=train, download=True, transform=transform)
        self.samples = [(img, 0 if lbl == _CAT else 1) for img, lbl in base if lbl in (_CAT, _DOG)]
    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

def balanced_subset(ds, n, offset=0):
    by_class = {0: [], 1: []}
    for i, (_, lbl) in enumerate(ds.samples): by_class[lbl].append(i)
    sel = []
    for c in [0, 1]: sel.extend(by_class[c][offset:offset+n])
    return Subset(ds, sel)

kw = dict(batch_size=BATCH_SIZE, num_workers=2)
train_loader = DataLoader(balanced_subset(CatDogCIFAR(True,  get_transform(True)),  TRAIN_N, 0),      shuffle=True,  **kw)
val_loader   = DataLoader(balanced_subset(CatDogCIFAR(True,  get_transform(False)), VAL_N,   TRAIN_N), shuffle=False, **kw)
test_loader  = DataLoader(balanced_subset(CatDogCIFAR(False, get_transform(False)), TEST_N,  0),      shuffle=False, **kw)
print(f"Train={TRAIN_N*2}  Val={VAL_N*2}  Test={TEST_N*2}")

## Model & Training

In [ ]:
def build_vgg16(pretrained):
    torch.manual_seed(SEED)
    m = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
    if pretrained:
        for p in m.features.parameters(): p.requires_grad = False
    m.classifier[6] = nn.Linear(4096, 2)
    return m.to(DEVICE)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, n = 0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(); out = model(x); loss = criterion(out, y)
        loss.backward(); optimizer.step()
        loss_sum += loss.item()*len(y); correct += (out.argmax(1)==y).sum().item(); n += len(y)
    return loss_sum/n, correct/n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, n = 0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE); out = model(x)
        loss_sum += criterion(out, y).item()*len(y); correct += (out.argmax(1)==y).sum().item(); n += len(y)
    return loss_sum/n, correct/n

## 실험 실행

In [ ]:
def run_experiment(name, pretrained):
    print(f"\n{'='*55}\n  {name}\n{'='*55}")
    model = build_vgg16(pretrained)
    criterion = nn.CrossEntropyLoss()
    _, pre_acc = evaluate(model, test_loader, criterion)
    print(f"  [Before Training] Test Acc: {pre_acc*100:.1f}%")
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    hist = {"train_acc": [], "val_acc": [], "val_loss": []}
    t0 = time.time()
    for ep in range(1, EPOCHS+1):
        _, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        hist["train_acc"].append(round(tr_acc, 4))
        hist["val_acc"].append(round(vl_acc, 4))
        hist["val_loss"].append(round(vl_loss, 4))
        print(f"  Epoch {ep}/{EPOCHS}  train_acc={tr_acc:.4f}  val_acc={vl_acc:.4f}  val_loss={vl_loss:.4f}")
    elapsed = time.time() - t0
    _, test_acc = evaluate(model, test_loader, criterion)
    print(f"\n  [After Training]  Test Acc: {test_acc*100:.1f}%  ({elapsed:.0f}s)")
    hist.update({"pre_test_acc": round(pre_acc, 4), "test_acc": round(test_acc, 4), "elapsed": round(elapsed, 1)})
    return hist

results = {}
results["Scratch"]  = run_experiment("Scratch (no pretrain)", pretrained=False)
results["Transfer"] = run_experiment("Transfer (pretrained)", pretrained=True)

## 결과 비교

In [ ]:
s, t = results["Scratch"], results["Transfer"]
print(f"{'':35s}  {'Scratch':>9}  {'Transfer':>9}")
print(f"  {'Test Acc (before)':35s}  {s['pre_test_acc']*100:>8.1f}%  {t['pre_test_acc']*100:>8.1f}%")
print(f"  {'Test Acc (after)':35s}  {s['test_acc']*100:>8.1f}%  {t['test_acc']*100:>8.1f}%")
print(f"  {'Training Time (s)':35s}  {s['elapsed']:>9.1f}  {t['elapsed']:>9.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, EPOCHS+1)
for name, hist, color in [("Scratch", s, "tomato"), ("Transfer", t, "steelblue")]:
    axes[0].plot(ep, hist["val_acc"],  label=name, color=color, marker="o")
    axes[1].plot(ep, hist["val_loss"], label=name, color=color, marker="o")
for ax, title, ylabel in zip(axes, ["Validation Accuracy", "Validation Loss"], ["Accuracy", "Loss"]):
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel); ax.legend(); ax.grid(alpha=0.3)
axes[0].set_ylim(0, 1)
plt.tight_layout(); plt.show()